In [1]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

In [2]:
df = pd.read_csv("../data/bank.csv")

In [3]:
df

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11157,33,blue-collar,single,primary,no,1,yes,no,cellular,20,apr,257,1,-1,0,unknown,no
11158,39,services,married,secondary,no,733,no,no,unknown,16,jun,83,4,-1,0,unknown,no
11159,32,technician,single,secondary,no,29,no,no,cellular,19,aug,156,2,-1,0,unknown,no
11160,43,technician,married,secondary,no,0,no,yes,cellular,8,may,9,2,172,5,failure,no


In [7]:
col_mapping = {
    'age': '연령',
    'job': '직업',
    'marital': '결혼상태',
    'education': '교육수준',
    'default': '채무불이행여부',
    'balance': '연간평균잔고',
    'housing': '주택담보대출여부',
    'loan': '개인신용대출여부',
    'contact': '연락수단',
    'day': '최근연락일',
    'month': '최근연락월',
    'duration': '통화지속시간',
    'campaign': '캠페인연락건수',
    'pdays': '이전연락후경과일',
    'previous': '이전연락건수',
    'poutcome': '이전캠페인결과',
    'deposit': '정기예금가입여부' 
}

In [8]:
job_mapping = {
    'admin.': '사무직', 'technician': '기술직', 'services': '서비스직', 
    'management': '관리직', 'retired': '은퇴자', 'blue-collar': '생산직', 
    'unemployed': '무직', 'entrepreneur': '개인사업자', 'housemaid': '가사도우미', 
    'unknown': '알수없음', 'self-employed': '자영업자', 'student': '학생'
}

In [9]:
marital_mapping = {'married': '기혼', 'single': '미혼', 'divorced': '이혼'}
education_mapping = {'primary': '초졸', 'secondary': '중고졸', 'tertiary': '대졸이상', 'unknown': '알수없음'}
yes_no_mapping = {'yes': '예', 'no': '아니오'}
contact_mapping = {'unknown': '알수없음', 'cellular': '휴대폰', 'telephone': '유선전화'}
month_mapping = {
    'jan': '1월', 'feb': '2월', 'mar': '3월', 'apr': '4월', 'may': '5월', 'jun': '6월',
    'jul': '7월', 'aug': '8월', 'sep': '9월', 'oct': '10월', 'nov': '11월', 'dec': '12월'
}
poutcome_mapping = {'unknown': '알수없음', 'other': '기타', 'failure': '실패', 'success': '성공'}

In [10]:
df.rename(columns=col_mapping, inplace=True)

In [11]:
df

,연령,직업,결혼상태,교육수준,채무불이행여부,연간평균잔고,주택담보대출여부,개인신용대출여부,연락수단,최근연락일,최근연락월,통화지속시간,캠페인연락건수,이전연락후경과일,이전연락건수,이전캠페인결과,정기예금가입여부
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11157,33,blue-collar,single,primary,no,1,yes,no,cellular,20,apr,257,1,-1,0,unknown,no
11158,39,services,married,secondary,no,733,no,no,unknown,16,jun,83,4,-1,0,unknown,no
11159,32,technician,single,secondary,no,29,no,no,cellular,19,aug,156,2,-1,0,unknown,no
11160,43,technician,married,secondary,no,0,no,yes,cellular,8,may,9,2,172,5,failure,no


In [12]:
df['직업'] = df['직업'].map(job_mapping)
df['결혼상태'] = df['결혼상태'].map(marital_mapping)
df['교육수준'] = df['교육수준'].map(education_mapping)
df['채무불이행여부'] = df['채무불이행여부'].map(yes_no_mapping)
df['주택담보대출여부'] = df['주택담보대출여부'].map(yes_no_mapping)
df['개인신용대출여부'] = df['개인신용대출여부'].map(yes_no_mapping)
df['연락수단'] = df['연락수단'].map(contact_mapping)
df['최근연락월'] = df['최근연락월'].map(month_mapping)
df['이전캠페인결과'] = df['이전캠페인결과'].map(poutcome_mapping)
df['정기예금가입여부'] = df['정기예금가입여부'].map(yes_no_mapping)

In [13]:
df

,연령,직업,결혼상태,교육수준,채무불이행여부,연간평균잔고,주택담보대출여부,개인신용대출여부,연락수단,최근연락일,최근연락월,통화지속시간,캠페인연락건수,이전연락후경과일,이전연락건수,이전캠페인결과,정기예금가입여부
0,59,사무직,기혼,중고졸,아니오,2343,예,아니오,알수없음,5,5월,1042,1,-1,0,알수없음,예
1,56,사무직,기혼,중고졸,아니오,45,아니오,아니오,알수없음,5,5월,1467,1,-1,0,알수없음,예
2,41,기술직,기혼,중고졸,아니오,1270,예,아니오,알수없음,5,5월,1389,1,-1,0,알수없음,예
3,55,서비스직,기혼,중고졸,아니오,2476,예,아니오,알수없음,5,5월,579,1,-1,0,알수없음,예
4,54,사무직,기혼,대졸이상,아니오,184,아니오,아니오,알수없음,5,5월,673,2,-1,0,알수없음,예
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11157,33,생산직,미혼,초졸,아니오,1,예,아니오,휴대폰,20,4월,257,1,-1,0,알수없음,아니오
11158,39,서비스직,기혼,중고졸,아니오,733,아니오,아니오,알수없음,16,6월,83,4,-1,0,알수없음,아니오
11159,32,기술직,미혼,중고졸,아니오,29,아니오,아니오,휴대폰,19,8월,156,2,-1,0,알수없음,아니오
11160,43,기술직,기혼,중고졸,아니오,0,아니오,예,휴대폰,8,5월,9,2,172,5,실패,아니오


In [20]:
df.columns

Index(['연령', '직업', '결혼상태', '교육수준', '채무불이행여부', '연간평균잔고', '주택담보대출여부', '개인신용대출여부',
       '연락수단', '최근연락일', '최근연락월', '통화지속시간', '캠페인연락건수', '이전연락후경과일', '이전연락건수',
       '이전캠페인결과', '정기예금가입여부'],
      dtype='object')

In [21]:
df = df.drop(columns=["통화지속시간", "최근연락월", "최근연락일","연락수단","교육수준"])

In [22]:
df.columns

Index(['연령', '직업', '결혼상태', '채무불이행여부', '연간평균잔고', '주택담보대출여부', '개인신용대출여부',
       '캠페인연락건수', '이전연락후경과일', '이전연락건수', '이전캠페인결과', '정기예금가입여부'],
      dtype='object')

In [23]:
df = df.drop(columns=["채무불이행여부", "캠페인연락건수", "이전연락후경과일","이전연락건수","이전캠페인결과"])

In [24]:
df.columns

Index(['연령', '직업', '결혼상태', '연간평균잔고', '주택담보대출여부', '개인신용대출여부', '정기예금가입여부'], dtype='object')

In [25]:
df

,연령,직업,결혼상태,연간평균잔고,주택담보대출여부,개인신용대출여부,정기예금가입여부
0,59,사무직,기혼,2343,예,아니오,예
1,56,사무직,기혼,45,아니오,아니오,예
2,41,기술직,기혼,1270,예,아니오,예
3,55,서비스직,기혼,2476,예,아니오,예
4,54,사무직,기혼,184,아니오,아니오,예
...,...,...,...,...,...,...,...
11157,33,생산직,미혼,1,예,아니오,아니오
11158,39,서비스직,기혼,733,아니오,아니오,아니오
11159,32,기술직,미혼,29,아니오,아니오,아니오
11160,43,기술직,기혼,0,아니오,예,아니오


In [26]:
df.to_csv("data_processed.csv", index=False,encoding="utf-8-sig")